Copyright (c) Technical University of Munich. All Rights Reserved.

<table>
<tr>
 <td><img src="https://raw.githubusercontent.com/CS4MS/CS4MS_exercise/main/images/logo_CS_MS_final.png" height=200 style="float: left;"></td>
    <td>
        <h1>Computer Science for Medical Students</h1>
        <h2>Exercise 8: Vision-Languag Model for Radiology</h2>
        <h3>Matthias Keicher &amp; Chantal Pellegrini</h3>
        <a href="https://www.cs.cit.tum.de/camp/research/vision-language/" target="_blank">Our Vision-Language Research Group @ CAMP</a>
        </br></br>
        <a href="https://github.com/CS4MS/" target="_blank">CS4MS GitHub Repository</a>
    </td>
</tr>
</table>

# Introduction



## Recap CLIP
![CLIP: Contrastive Language-Image Pretraining](https://github.com/openai/CLIP/raw/main/CLIP.png)

[OpenAI blog post with more details](https://openai.com/research/clip)

## BioVil: Making the Most of Text Semantics to Improve Biomedical Vision–Language Processing

![BioVil architecture](https://www.microsoft.com/en-us/research/uploads/prod/2022/07/BioVIL-1024x328.png)
- Contrastive language-image pretraining on chest X-rays and corresponding radiology reports proposed by Microsoft Research [1].
- Trained on the [MIMIC-CXR dataset](https://physionet.org/content/mimic-cxr/2.0.0/) with 377,110 radiographs and corresponding radiology reports.

- [BioVil blog post with more details](https://www.microsoft.com/en-us/research/publication/making-the-most-of-text-semantics-to-improve-biomedical-vision-language-processing/)

[1] Boecking, Benedikt, et al. "Making the most of text semantics to improve biomedical vision–language processing." European conference on computer vision. Cham: Springer Nature Switzerland, 2022.

<br >

### Image Encoder
![CNN](https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png)
![ResNet50](https://upload.wikimedia.org/wikipedia/commons/9/98/ResNet50.png)

- [ResNet50](https://openaccess.thecvf.com/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html) CNN architecture
- Pretrained on chest X-rays with contrastive pretraining ([SimCLR](http://proceedings.mlr.press/v119/chen20j.html))

<br >

### Text Encoder

![BioVil CXR-BERT Text Encoder](https://www.microsoft.com/en-us/research/uploads/prod/2022/07/CXR-BERT.png)

- Based on [BERT Transformer encoder architecture](https://ai.googleblog.com/2018/11/open-sourcing-bert-state-of-art-pre.html)
- Pretrained on PubMed articles, [MIMIC clinical notes](https://physionet.org/content/mimiciii) and MIMIC-CXR radiology reports


# Vision-Language Model
Installation of dependencies and loading of text and image encoders

## Dependencies

In [ ]:
# Installation of libraries needed for transformer-based language models and timm for CNN image encoders
%pip install "transformers<5.0" --quiet
%pip install timm accelerate --quiet

## Loading Image Encoder

In [ ]:
from utils import ImageEncoderWeightTypes, load_biovil_image_components


Load model and move to GPU

In [ ]:
image_encoder, image_transforms, device, get_image_embeddings = load_biovil_image_components(
    ImageEncoderWeightTypes.BIOVIL_T
)


## Loading Text Encoder

In [ ]:
from utils import load_biovil_text_components

tokenizer, text_encoder, get_text_embeddings = load_biovil_text_components(device)


## Helper functions

### Cosine Similarity
The cosine similarity is a similarity measure between two vectors that only compares the similarity of their direction, not their length. Mathematically the dot product is calculated between the normalized vectors.

More information on wikipedia: https://en.wikipedia.org/wiki/Cosine_similarity

In [ ]:
from utils import calculate_cosine_similarity


### Download image
Helper method to download an image from a given url.

In [ ]:
from utils import load_image

# test with a random sample from the Indiana chest x-ray dataset
url = 'https://openi.nlm.nih.gov/imgs/512/276/677/CXR677_IM-2249-1001.png?keywords=Catheters,%20Indwelling,Lung,Density,Density,Density,Pleural%20Effusion,Pneumonia'
print('downloaded url:', url)
display(load_image(url))


### Plot results
This is another helper method to visualize the cosine similarities between text and images as well as softmax probabilities.

In [ ]:
from utils import plot_similarities


# Downstream Tasks

## Test patients - Indiana University, Chest X-rays

Patient class

In [ ]:
from utils import Patient


### Patient Dataset

In [ ]:
import torch

# dictionary with patients
patients = {}

patients['healthy'] = Patient(
    image_url = 'https://openi.nlm.nih.gov/imgs/512/393/3200/CXR3200_IM-1512-1001.png?keywords=normal',
    report = 'Heart size is normal and the lungs are clear.'
)

patients['pneumonia'] = Patient(
    image_url = 'https://openi.nlm.nih.gov/imgs/512/276/677/CXR677_IM-2249-1001.png?keywords=Catheters,%20Indwelling,Lung,Density,Density,Density,Pleural%20Effusion,Pneumonia',
    report = 'PICC line catheter tip XXXX in the right atrium. Heart is not enlarged. Trachea and XXXX bronchi appear normal. Lungs are mildly under expanded. No pneumothorax. There are small areas of patchy density in the left lower lung XXXX. There is a larger area of XXXX patchy density in the right mid and lower lungs with right-sided pleural effusion.',
)

patients['atelectasis'] = Patient(
    image_url = 'https://openi.nlm.nih.gov/imgs/512/242/1445/CXR1445_IM-0287-4004.png?keywords=Diaphragm,Pulmonary%20Atelectasis,Consolidation,Pleural%20Effusion,Catheters,%20Indwelling,Tube,%20Inserted,Airspace%20Disease',
    report = 'Stable cardiomediastinal silhouette. There has been interval removal of right chest tube with increased elevation of the right hemidiaphragm and XXXX right basilar atelectasis. Left basilar consolidation and pleural effusions seen. No XXXX focal consolidation or pneumothorax. There is a stable left PICC with tip overlying the mid SVC and large XXXX feeding tube courses below the diaphragm.'
)

patients['cardiomegaly'] = Patient(
    image_url = 'https://openi.nlm.nih.gov/imgs/512/309/1111/CXR1111_IM-0077-4004.png?keywords=Technical%20Quality%20of%20Image%20Unsatisfactory%20,Cardiomegaly',
    report = 'Lordotic projection and large body habitus. Limited mediastinal evaluation. Severe cardiomegaly. No visualized pneumothorax. No large effusion or airspace disease. No fracture.',
)

patients['Nodules'] = Patient(
    image_url = 'https://openi.nlm.nih.gov/imgs/512/22/1626/CXR1626_IM-0407-1001.png?keywords=Nodule,Nodule',
    report = 'The heart is normal in size. The mediastinal contours are within normal limits. There are numerous bilateral pulmonary nodules of varying sizes. The largest is noted in the left lower lobe, posteriorly measuring approximately 7.0 cm. No acute infiltrate or pleural effusion are appreciated.'
)

for i, (description, patient) in enumerate(patients.items(), 1):
    print(f'Patient {i}:', description)
    display(patient)

patient_images = [patient.image for patient in patients.values()]
patient_reports = [patient.report for patient in patients.values()]
patient_descriptions = [description for description in patients.keys()]

## Zero-shot X-Ray Classification

### Contrastive binary classification
The basic idea of contrastive zero-shot classification is to encode both a positive and negative description (e.g. presence and absence) of a class to be predicted. Next, the image is encoded in the same space and then evaluated if it is closer to the positive or negative text embedding. The [softmax function](https://en.wikipedia.org/wiki/Softmax_function) allows us to estimate a probability for this prediction.

In [ ]:
from utils import build_similarity_function

get_similarities_from_text_and_images = build_similarity_function(get_text_embeddings, get_image_embeddings)


#### Basic Prompting

##### Healthy

In [ ]:
text_prompts = [
    'healthy',
    'not healthy',
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Cardiomegaly

In [ ]:
text_prompts = [
    'cardiomegaly',
    'no cardiomegaly',
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Pleural effusion

In [ ]:
text_prompts = [
    'pleural effusion',
    'no pleural effusion',
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Pneumonia

In [ ]:
text_prompts = [
    'pneumonia',
    'no pneumonia',
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Atelectasis

In [ ]:
text_prompts = [
    'atelectasis',
    'no atelectasis',
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

#### Report Style Prompting

In [ ]:
# dictionary to save prompts
prompts = {}

##### Healthy

In [ ]:
prompts['healthy'] = '''
The lungs are clear.
Normal heart size and shape.
No abnormal fluid buildup.
No visible tumors or masses. No pneumothorax.
'''

prompts['not healthy'] = '''
There is an area of increased opacity and consolidation indicating pneumonia.
Enlargement of the heart silhouette indicating cardiomegaly.
There is a loss in volume indicating atelectasis.
'''

text_prompts = [
    prompts['healthy'],
    prompts['not healthy']
]
similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Cardiomegaly

In [ ]:
prompts['cardiomegaly'] = '''
Increased size of the heart shadow.
Enlargement of the heart silhouette.
Increased diameter of the heart border.
Increased cardiothoracic ratio.
'''
prompts['no cardiomegaly'] = '''
The heart shadow size is normal.
The heart silhouette is normal.
Normal diameter of the heart border.
Normal cardiothoracic ratio.
'''

text_prompts = [
    prompts['cardiomegaly'],
    prompts['no cardiomegaly']
]

similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Pleural Effusion

In [ ]:
prompts['pleural effusion'] = '''
Blunting of costophrenic angles.
Opacity in the lower lung fields.
Mediastinal shift.
Reduced lung volume.
Presence of meniscus sign or veil-like appearance.
'''

prompts['no pleural effusion'] = '''
No blunting of costophrenic angles.
No opacity in the lower lung fields.
The lungs are clear.
No mediastinal shift.
No presence of meniscus sign or veil-like appearance.
'''

text_prompts = [
    prompts['pleural effusion'],
    prompts['no pleural effusion']
]

similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Pneumonia

In [ ]:
prompts['pneumonia'] = 'There is an area of increased opacity and consolidation indicating pneumonia.'
prompts['no pneumonia'] = 'There are no opacities, no consolidation and no pleural effusion. No signs of pneumonia.'

text_prompts = [
    prompts['pneumonia'],
    prompts['no pneumonia']
]

similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

##### Atelectasis

In [ ]:
prompts['atelectasis'] = '''
The lung appears reduced in volume supporting the presence of atelectasis.
The mediastinum shows a shift, consistent with volume loss associated with atelectasis.
These findings are consistent with atelectasis.
'''

prompts['no atelectasis'] = '''
Both lungs are well-expanded with clear lung fields.
The lung volumes appear normal and symmetric, with no apparent reduction in the size of either lung.
The mediastinum is positioned centrally, without evidence of mediastinal shift.
No radiographic signs of atelectasis are present. The lungs appear normally aerated and expanded.
'''

text_prompts = [
    prompts['atelectasis'],
    prompts['no atelectasis']
]

similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranging from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

## Stanardized Report Generation

#### Structured reporting template definition

In [ ]:
reporting_template = {
        'healthy' : 'The lungs are clear. No findings.',
        'not healthy' :
            [
                {
                    'no cardiomegaly' : 'The heart is normal in size.',
                    'cardiomegaly' : 'There is cardiomegaly.',
                },
                {
                    'no pleural effusion': 'There is no pleural effusion.',
                    'pleural effusion': 'There is pleural effusion.'
                },
            ],
}

#### Report generation

In [ ]:
from utils import generate_report


In [ ]:
# generate report for all patients

for i, image in enumerate(patient_images, 1):
    print('Patient', i)
    display(image)
    report = ' '.join(generate_report(image, reporting_template, prompts, get_similarities_from_text_and_images))
    print('Generated report:', report,'\n\n')


#### Open tasks

1.  Why is cardiomegaly detected in patient 3 even though it is not present? How could this be fixed?

2. Change the prompts and observe the change in similarities

3. Add a new classification task e.g. lung opacity and come up with a report style prompt (try if ChatGPT can come up with good descriptors)

2. Extend the reporting with more choices, e.g. adding diagnoses or the severity assessment of cardiomegaly

# Additional Resources

## Radiology Report retrieval

Given a large database of reports the embedding of an image can be used to retrieve the reports most similar to the given image. In this example the images are compared to their matching reports:

In [ ]:
# write all ground truth reports in a list as text prompts
text_prompts = [report.replace('. ', '.\n') for report in patient_reports]

similarities = get_similarities_from_text_and_images(text_prompts, patient_images)
print('Cosine similarities ranging from -1 to 1:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=False)
print('\nSoftmax probabilities ranginf from 0% to 100%:')
plot_similarities(similarities, text_prompts, patient_images, patient_descriptions, plot_probabilities=True)

## Phrase grounding

BioVil allows for the visualization of text similarity in images referred to as "phrase grounding" [1]:

![BioVil phrase grounding examples](https://www.microsoft.com/en-us/research/uploads/prod/2022/07/MS-CXR-2048x472.png)

[Link to phrase grounding notebook](https://github.com/microsoft/hi-ml/blob/main/hi-ml-multimodal/notebooks/phrase_grounding.ipynb)

[BioVil code on github](https://github.com/microsoft/hi-ml/tree/main/hi-ml-multimodal)


## Xplainer: From X-Ray Observations to Explainable Zero-Shot Diagnosis

![Xplainer graphical abstract](https://raw.githubusercontent.com/ChantalMP/Xplainer/master/figures/model_overview.png)

We propose a new way of explainability for zero-shot diagnosis prediction in the clinical domain. Instead of directly predicting a diagnosis, we prompt the model to classify the existence of descriptive observations, which a radiologist would look for on an X-Ray scan, and use the descriptor probabilities to estimate the likelihood of a diagnosis, making our model explainable by design. For this we leverage BioVil, a pretrained CLIP model for X-rays and apply contrastive observation-based prompting. We evaluate Xplainer on two chest X-ray datasets, CheXpert and ChestX-ray14, and demonstrate its effectiveness in improving the performance and explainability of zero-shot diagnosis.

Pellegrini, Chantal, et al. "Xplainer: From X-Ray Observations to Explainable Zero-Shot Diagnosis." accepted at MICCAI 2023.

[MICCAI 2023 Paper](https://link.springer.com/chapter/10.1007/978-3-031-43904-9_41)

[Huggingface Demo](https://huggingface.co/spaces/CAMP-ViL/Xplainer)

[Preprint on arxiv](https://arxiv.org/abs/2303.13391)

[Code on GitHub](https://github.com/ChantalMP/Xplainer)
